# Notebook 1: Pre-compute Event Embeddings with Qwen3-4B

04/11: run with single embedding vector output.

In [1]:
!pip install -q transformers accelerate torch

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_token = user_secrets.get_secret("HF_TOKEN")

login(token=HF_token)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [5]:
event_df_path = "/kaggle/input/notebooks/zygong1994/qwen-3-filter-gdelt-event/gdelt_events_with_market_impact.csv"
market_df_path = "/kaggle/input/datasets/zygong1994/finance-world-model-dataset/combined_commodity_data.csv"

## Load Data

In [6]:
events_df = pd.read_csv(event_df_path)
events_df["event_time"] = pd.to_datetime(events_df["event_time"])
events_df = events_df[events_df["market_impact"] == 1].reset_index(drop=True)
print(f"Market-relevant events: {len(events_df)}")
events_df.head(3)

Market-relevant events: 803


,event_year,rank_in_year,GLOBALEVENTID,SQLDATE,Actor1Name,Actor1Type1Code,Actor1CountryCode,Actor2Name,Actor2Type1Code,Actor2CountryCode,...,GoldsteinScale,NumArticles,AvgTone,impact_score,SOURCEURL,event_summary,key_actors,event_time,fetch_success,market_impact
0,2015,1,435199599,20150522,IRAQI,GOV,IRQ,SHIA,UAF,NaN,...,-7.2,6,-53.962901,19.652643,http://freerepublic.com/focus/f-news/3292340/p...,Thousands of Shia militia fighters arrived at ...,"['IRAQI GOVERNMENT', 'SHIA MILITIAS', 'ISIS']",2015-05-22,False,1
1,2015,5,436606953,20150527,SINAI,NaN,EGY,POLICEMAN,COP,NaN,...,-10.0,20,-30.000000,13.913357,https://www.blesk.cz/clanek/zpravy-live-zpravy...,An Egyptian policeman was killed and eight oth...,"['Egyptian Policeman', 'Sinai Province', 'Egyp...",2015-05-27,False,1
2,2015,6,473766932,20151007,NIGERIA,UAF,NGA,MILITANT,UAF,NaN,...,-10.0,10,-30.555556,13.886035,http://www.timeturk.com/nijerya-da-catisma-50-...,The Nigerian military killed 50 Boko Haram mil...,"['Nigerian Army', 'Boko Haram militants']",2015-10-07,False,1


In [7]:
market_df = pd.read_csv(market_df_path)
market_df.index = pd.to_datetime(market_df["date"])
feature_cols = [c for c in market_df.columns if c != "date"]
print(f"Market data: {market_df.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols[:5]}...")

market_df.head(3)

Market data: (4118, 47)
Features (46): ['YF_Price_Gold', 'YF_Price_Silver', 'YF_Price_Crude Oil (WTI)', 'YF_Price_Crude Oil (Brent)', 'YF_Price_Natural Gas']...


,date,YF_Price_Gold,YF_Price_Silver,YF_Price_Crude Oil (WTI),YF_Price_Crude Oil (Brent),YF_Price_Natural Gas,YF_Price_Copper,YF_Price_Iron Ore,YF_Price_Lithium,YF_Price_Wheat,...,FRED_PCEPILFE,FRED_PPIACO,FRED_UNRATE,FRED_PAYEMS,FRED_ICSA,FRED_DEXCHUS,EIA_Crude Inventory (US),EIA_Strategic Petroleum Reserve,EIA_Natural Gas Stock,EIA_US Crude Production
date,,,,,,,,,,,,,,,,,,,,,
2014-12-20,2014-12-20,1179.699951,15.649,55.259998,60.110001,3.144,2.9035,20.592468,19.356659,625.75,...,96.214,192.0,5.7,140568.0,276000.0,6.2213,1043942.0,690963.0,3220.0,9121.0
2014-12-21,2014-12-21,1179.699951,15.649,55.259998,60.110001,3.144,2.9035,20.592468,19.356659,625.75,...,96.214,192.0,5.7,140568.0,276000.0,6.2213,1043942.0,690963.0,3220.0,9121.0
2014-12-22,2014-12-22,1179.699951,15.649,55.259998,60.110001,3.144,2.9035,20.592468,19.356659,625.75,...,96.214,192.0,5.7,140568.0,276000.0,6.2213,1043942.0,690963.0,3220.0,9121.0


## Format Date-Level Texts (Market Context + All Events That Day)

In [8]:
def format_date_text(date, events_df, market_df, feature_cols):
    """Build a single text for a date: market context + all events that day.

    Args:
        date: the target date (pd.Timestamp or str)
        events_df: DataFrame with 'event_time' and 'event_summary' columns
        market_df: market DataFrame indexed by date
        feature_cols: list of numeric feature column names

    Returns:
        Formatted text string for this date.
    """
    date = pd.Timestamp(date)
    date_str = date.strftime("%Y-%m-%d")

    # --- Market context ---
    available = market_df.index[market_df.index <= date]
    if len(available) == 0:
        market_date = market_df.index[0]
    else:
        market_date = available[-1]
    market_row = market_df.loc[market_date]

    parts = []
    for col in feature_cols:
        val = market_row[col]
        if pd.isna(val):
            continue
        short_name = col.replace("YF_Price_", "").replace("YF_Volume_", "Vol_").replace("FRED_", "")
        if "Price" in col or "SP500" in col:
            parts.append(f"{short_name} ${val:.2f}")
        elif "DGS" in col or "DFF" in col or "T10YIE" in col:
            parts.append(f"{short_name} {val:.2f}%")
        elif "Volume" in col:
            parts.append(f"{short_name} {val:.0f}")
        else:
            parts.append(f"{short_name} {val:.2f}")
    context = ", ".join(parts)

    # --- Events on this date ---
    day_events = events_df[events_df["event_time"].dt.date == date.date()]
    event_lines = []
    for _, row in day_events.iterrows():
        event_lines.append(f"- {row['event_summary']}")

    if event_lines:
        events_block = "\n".join(event_lines)
        return f"Market on {date_str}: {context}\nEvents ({date_str}):\n{events_block}"
    else:
        return f"Market on {date_str}: {context}\nNo significant events."


# Generate texts for ALL dates from 2015-01-01 to 2016-12-31
all_dates = pd.date_range("2015-01-01", "2016-12-31", freq="D")
date_texts = {}
for d in all_dates:
    date_str = d.strftime("%Y-%m-%d")
    date_texts[date_str] = format_date_text(d, events_df, market_df, feature_cols)

event_dates = set(events_df["event_time"].dt.strftime("%Y-%m-%d"))
n_with = sum(1 for d in date_texts if d in event_dates)
n_without = len(date_texts) - n_with
print(f"Total dates: {len(date_texts)} ({n_with} with events, {n_without} without)")

# Show examples
sample_with = next(d for d in date_texts if d in event_dates)
sample_without = next(d for d in date_texts if d not in event_dates)
print(f"\n--- With events ({sample_with}) ---\n{date_texts[sample_with]}")
print(f"\n--- Without events ({sample_without}) ---\n{date_texts[sample_without]}")

Total dates: 731 (96 with events, 635 without)

--- With events (2015-02-19) ---
Market on 2015-02-19: Gold $1207.10, Silver $16.37, Crude Oil (WTI) $51.16, Crude Oil (Brent) $60.21, Natural Gas $2.83, Copper $2.64, Iron Ore $21.92, Lithium $20.34, Wheat $527.75, Soybean $1007.25, YF_State_VXD(Dow Volatility Index) 14.97, YF_State_VXN(Nasdaq Volatility Index) 15.44, YF_State_GVZ(Gold Volatility Index) 18.62, YF_State_EVZ(Energy Volatility Index) 11.93, YF_State_MSCI World Index 60.05, YF_State_MSCI All Country Index 48.81, YF_State_Nikkei 225 18264.79, YF_State_EURO STOXX 50 3488.08, YF_State_Shanghai Composite 3240.77, YF_State_Hang Seng Index 24817.91, DGS10 2.11%, T10YIE 1.74%, DTWEXBGS 105.14, INDPRO 102.23, DFF 0.12%, SP500 $2037.05, DJIA 17535.39, NASDAQCOM 4924.70, VIXCLS 15.29, OVXCLS 55.75, GDPC1 18666.62, A191RL1Q225SBEA 3.60, RSAFS 427119.00, CPIAUCSL 235.34, CPILFESL 240.17, PCEPI 96.83, PCEPILFE 96.32, PPIACO 191.10, UNRATE 5.50, PAYEMS 140827.00, ICSA 285000.00, DEXCHUS 6

## Load Qwen3-4B

In [4]:
MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
# Left-pad so the last position is always the real last token (simplifies pooling)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()
print(f"Model loaded. Hidden size: {model.config.hidden_size}")
print(f"vocab_size: {model.config.vocab_size}, pad_token_id: {tokenizer.pad_token_id}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded. Hidden size: 2560
vocab_size: 151936, pad_token_id: 151643


## Extract Single-Vector Embeddings (One Pooled Vector per Date)

For each date-level text, take the last non-padding token's hidden state as a single pooled vector. This matches the single-vector Stage 1 pipeline (`02_stage1_single_vector.ipynb`) — no token-length padding issues downstream.

In [ ]:
BATCH_SIZE = 16

# Ordered lists for batching
all_date_strs = list(date_texts.keys())
all_texts = [date_texts[d] for d in all_date_strs]

embeddings_by_date = {}  # date_str -> tensor(1, hidden_size)

for start in range(0, len(all_texts), BATCH_SIZE):
    batch_texts = all_texts[start : start + BATCH_SIZE]
    batch_dates = all_date_strs[start : start + len(batch_texts)]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    # Sanity check on the first batch: input ids must be within vocab
    if start == 0:
        max_id = inputs["input_ids"].max().item()
        assert max_id < model.config.vocab_size, (
            f"input_id {max_id} >= vocab_size {model.config.vocab_size}"
        )
        print(f"First batch OK: max_input_id={max_id}, shape={tuple(inputs['input_ids'].shape)}")

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    last_hidden = outputs.hidden_states[-1]  # (batch, padded_seq_len, hidden_size)

    # With left-padding, the last position is always the real last token
    pooled = last_hidden[:, -1, :]  # (batch, hidden_size)

    # Move to CPU once per batch (cheaper than per-sample)
    pooled_cpu = pooled.float().cpu()
    for i, date_str in enumerate(batch_dates):
        # Shape (1, hidden): Stage 1 dataset treats axis 0 as "events/vectors for this date"
        embeddings_by_date[date_str] = pooled_cpu[i].unsqueeze(0).clone()

    if (start // BATCH_SIZE) % 10 == 0:
        print(f"Processed {start + len(batch_texts)}/{len(all_texts)} dates")

print(f"\nDone. Extracted {len(embeddings_by_date)} date-level pooled vectors.")
example = next(iter(embeddings_by_date.values()))
print(f"Per-date embedding shape: {tuple(example.shape)}  (1 x hidden)")

## Save Embeddings

In [ ]:
output = {
    "embeddings_by_date": embeddings_by_date,  # dict: "YYYY-MM-DD" -> tensor(1, hidden_size)
    "hidden_size": model.config.hidden_size,
    "model_id": MODEL_ID,
    "n_dates": len(embeddings_by_date),
    "pooling": "last_non_pad_token",
}

torch.save(output, "event_embeddings.pt")
print(f"Saved event_embeddings.pt")
print(f"  Dates: {len(embeddings_by_date)}")
print(f"  Hidden size: {model.config.hidden_size}")
ex = all_date_strs[0]
print(f"  Example: {ex} -> {tuple(embeddings_by_date[ex].shape)}  (1 x hidden)")